# IELTS Listening — Evaluator QLoRA fine-tune (Kaggle)

Fine-tunes the doc's **separate evaluator** (a Qwen2.5-**7B**-Instruct
LoRA — the doc permits 7B here for efficiency) to judge one answer at a
time:

> Input: Question + Official Answer + Accepted Variants + Student Answer
> Output: verdict / reason / correct_answer / skill

### Before you run
1. `cd backend && python tools/build_dataset.py` -> produces
   `data/datasets/evaluator_sft.jsonl`.
2. Upload it in the same Kaggle Dataset (`ielts-listening-sft`).
3. Accelerator = **GPU T4** (7B fits comfortably); Internet = On.

## 0. Get the repo + datasets

In [ ]:
# Cell 0 — pull this repo so the SFT datasets are on the Kaggle filesystem.
# This is a PRIVATE repo, so give Kaggle a token: Add-ons -> Secrets, add
# GITHUB_TOKEN = a PAT with read access to the repo (fine-grained 'Contents:
# read', or a classic token with the 'repo' scope).
# Alternative: skip cloning and add the 'ielts-listening-sft' Kaggle Dataset;
# the data cell below finds the jsonl either way.
import os
REPO_URL = "https://github.com/asiralabi/IELTS.git"
try:
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    REPO_URL = REPO_URL.replace("https://", f"https://{_tok}@")
except Exception:
    print("No GITHUB_TOKEN secret - trying anonymous clone "
          "(only works if the repo is public).")
if not os.path.isdir("ielts"):
    !git clone --depth 1 $REPO_URL ielts
!ls -lh ielts/backend/data/datasets/*.jsonl

In [ ]:
%%capture
# Unsloth gives ~2x faster QLoRA. Two Kaggle-specific gotchas: (1) the free
# Unsloth build trains on ONE GPU only, and (2) it needs a Turing-or-newer
# card — use a **T4** (compute capability 7.5). The P100 is Pascal (6.0) and
# has NO compiled Unsloth/Triton kernels -> 'no kernel image is available'.
# Fit depends on the model AND the corpus length: the generator's records
# are long (~5-7k tokens each), so it uses a 3B base; a 7B/14B OOMs a T4 at
# train time on that corpus (see the generator's section 2 for the math).
# If this install ever breaks, follow the current Kaggle snippet at
# https://github.com/unslothai/unsloth (the API used below is stable).
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes

## 1. Load Qwen2.5-7B in 4-bit

In [ ]:
import os
# Set BEFORE the torch/unsloth CUDA import. Lets the allocator grow
# segments instead of failing on a large contiguous request — cheap VRAM
# hygiene that reduces fragmentation-driven OOMs.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# Pin to ONE GPU. Unsloth's free build trains on a single GPU anyway, but
# if TWO are visible (e.g. you picked 'T4 x2') it loads under an accelerate
# device_map dispatch whose per-forward hooks pile extra tensors onto GPU 0
# and OOM it. One visible GPU = clean single-device load. Set before import.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import gc, torch
# Reclaim VRAM left by a PREVIOUS run in this kernel. If you re-run cells
# without restarting, the old 4-bit model + optimizer stay resident on the
# GPU and cause a misleading OOM at train time even though the model fits
# fresh. A clean Restart & Run All avoids this; this guard makes a reused
# kernel non-fatal by freeing those globals first.
for _n in ("model", "tokenizer", "trainer", "trainer_stats"):
    globals().pop(_n, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

from unsloth import FastLanguageModel

MODEL = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LEN = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,          # auto: bf16 on Ampere+, fp16 on T4
    load_in_4bit=True,   # QLoRA
    # Pin the whole model to GPU 0: no cross-GPU sharding, and the model
    # device matches the trainer device. A bnb 4-bit model loaded on a
    # different device than the trainer makes accelerate raise ValueError
    # ('can't train a model loaded in 4-bit precision on a different
    # device...') at trainer.train(); this is the fix it recommends.
    device_map={"": 0},
)

## 2. Attach LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",   # long scripts -> save VRAM
    random_state=3407,
)

## 3. Load the SFT dataset

Evaluator prompts are short, so `MAX_SEQ_LEN=1024` is plenty and keeps
training fast.

In [ ]:
import glob, os
from datasets import load_dataset

# Resolves the SFT jsonl produced by backend/tools/build_dataset.py.
# Works whether you git-clone this repo into the notebook OR add it as a
# Kaggle Dataset named 'ielts-listening-sft'.
FILENAME = "evaluator_sft.jsonl"
CANDIDATES = [
    f"/kaggle/input/ielts-listening-sft/{FILENAME}",
    *glob.glob(f"/kaggle/**/backend/data/datasets/{FILENAME}", recursive=True),
    *glob.glob(f"**/backend/data/datasets/{FILENAME}", recursive=True),
]
DATA_PATH = next((p for p in CANDIDATES if os.path.exists(p)), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        f"{FILENAME} not found. Git-clone this repo (see cell 0) or add the "
        "Kaggle Dataset 'ielts-listening-sft'.")
print("Using dataset:", DATA_PATH)

raw = load_dataset("json", data_files=DATA_PATH, split="train")

def to_text(row):
    # Each row is {"messages": [system, user, assistant]}; render with
    # the model's own chat template so training matches inference.
    return {"text": tokenizer.apply_chat_template(
        row["messages"], tokenize=False, add_generation_prompt=False)}

dataset = raw.map(to_text, remove_columns=raw.column_names)
print(dataset)
print(dataset[0]["text"][:600])

## 4. Train (response-only loss)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=1024,
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=5,
        num_train_epochs=2,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="qwen2.5-7b-ielts-evaluator-lora-checkpoints",
        report_to="none",
    ),
)

# Mask the prompt: only the assistant JSON contributes to the loss, so
# the model learns to PRODUCE exams, not to echo the spec/instructions.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

In [ ]:
trainer_stats = trainer.train()
print(trainer_stats)

## 5. Save adapter + GGUF

In [ ]:
# 1) LoRA adapter only (tiny, ~100-300 MB) — load on top of the base.
model.save_pretrained("qwen2.5-7b-ielts-evaluator-lora")
tokenizer.save_pretrained("qwen2.5-7b-ielts-evaluator-lora")

# 2) GGUF (q4_k_m) for llama.cpp / Ollama serving on CPU or small GPU.
#    Merges + quantises; needs several GB of /kaggle/working disk.
model.save_pretrained_gguf("qwen2.5-7b-ielts-evaluator-gguf", tokenizer, quantization_method="q4_k_m")

# 3) (optional) merged 16-bit HF weights for vLLM (~6 GB for 3B, ~14 GB for
#    7B) — uncomment
#    only if you have the disk / will push to the HF Hub.
# model.save_pretrained_merged("qwen2.5-7b-ielts-evaluator-lora-merged16", tokenizer, save_method="merged_16bit")

## 6. Serve it as a second model

Serve the GGUF as its own Ollama model:
```
ollama create ielts-evaluator -f Modelfile   # FROM the evaluator GGUF
```
The evaluator is a **separate** model from the generator, so using it
for marking needs a small backend addition: a second client pointed at
`ielts-evaluator` that runs the `EVALUATOR_SYSTEM` prompt
(`app/llm/prompts.py`) per answer, feeding results into
`listening_trainer.check_full_test`. Until that wiring lands, the base
model + `ANSWER_CHECKER_SYSTEM` continues to mark whole sets. The
system prompt this model was trained on lives in `EVALUATOR_SYSTEM`.